In [ ]:
!pip install sentence-transformers faiss-gpu-cu11

In [ ]:
import ir_datasets
from tqdm import tqdm
import ir_datasets
from itertools import islice
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [ ]:
def preview_iter(title, iterator, fields, n=5):
    print(f"\n========== {title} ==========")

    for i, item in enumerate(islice(iterator, n), start=1):
        print(f"\n--- {title[:-1].title()} {i} ---")
        for field in fields:
            value = getattr(item, field)
            if field == "text":
                value = value[:500]
            print(f"{field}:", value)


def download_and_preview_msmarco(
    corpus_id="msmarco-passage",
    eval_id="msmarco-passage/dev/small",
    n_samples=1,
):
    passages = ir_datasets.load(corpus_id)
    eval_set = ir_datasets.load(eval_id)

    preview_iter(
        "SAMPLE PASSAGES",
        passages.docs_iter(),
        fields=["doc_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QUERIES",
        eval_set.queries_iter(),
        fields=["query_id", "text"],
        n=n_samples,
    )

    preview_iter(
        "SAMPLE QRELS",
        eval_set.qrels_iter(),
        fields=["query_id", "doc_id", "relevance"],
        n=n_samples,
    )

    corpus = {}
    queries = {}
    qrels = {}

    print("\n========== LOADING FULL CORPUS ==========")
    for doc in tqdm(passages.docs_iter(), desc="Loading passages"):
        corpus[doc.doc_id] = doc.text

    print(f"Total passages loaded: {len(corpus):,}")

    print("\n========== LOADING QUERIES ==========")
    for query in tqdm(eval_set.queries_iter(), desc="Loading queries"):
        queries[query.query_id] = query.text

    print(f"Total queries loaded: {len(queries):,}")

    print("\n========== LOADING QRELS ==========")
    for qrel in tqdm(eval_set.qrels_iter(), desc="Loading qrels"):
        qrels.setdefault(qrel.query_id, {})[qrel.doc_id] = qrel.relevance

    print(f"Total qrels queries loaded: {len(qrels):,}")

    return corpus, queries, qrels


corpus, _, __ = download_and_preview_msmarco()

In [ ]:
import json
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer


model_name = "BAAI/bge-small-en-v1.5"
save_emb_path = "artifacts/embeddings.npy"
save_meta_path = "artifacts/doc_ids.jsonl"

model = SentenceTransformer(model_name)

doc_ids = []
texts = []

# Ví dụ nếu bạn dùng ir_datasets
for doc in corpus.docs_iter():
    doc_ids.append(doc.doc_id)
    texts.append(doc.text)

embeddings = model.encode(
    texts,
    batch_size=256,
    device=["cuda:0", "cuda:1"],
    normalize_embeddings=True,
    show_progress_bar=True,
)

embeddings = embeddings.astype("float32")

np.save(save_emb_path, embeddings)

with open(save_meta_path, "w", encoding="utf-8") as f:
    for i, doc_id in enumerate(doc_ids):
        f.write(json.dumps({
            "row_id": i,
            "doc_id": doc_id,
        }, ensure_ascii=False) + "\n")

print("Saved embeddings:", embeddings.shape)